# P4 / 01: is daily reversal stronger after removing industry moves and earnings news? (both windows)

Protocol: [`001_claude_p4_news_filtered_reversal_preregistration.md`](../../discussions/News_Filtered_Reversal/001_claude_p4_news_filtered_reversal_preregistration.md), section 5. Paper: Da, Liu & Schaumburg (2014).

Arms (quintile reversal, long the bottom / short the top, overlapping h = k, one-session skip, close-slot measured costs):
- **R** raw past-k return;
- **I** industry-adjusted (minus the sector ETF);
- **IN** I without stocks that had an earnings 8-K window {f−1, f, f+1} in the formation period.
- Secondary (reported): **IA** I without Alpaca news shocks; **INA** without both.

**Gates** per window and horizon (k = 21 primary, 5; α = 0.025):
- **F1** IN − I > 0 (bootstrap, and against 200 random exclusions of the same size);
- **F2** I − R > 0;
- **F3** IN net > 0 at 1× (bootstrap) and 2× (mean).

`REPLICATES` needs F1 and F2 in **both** windows.

In [ ]:
RUN_NEWS = True          # False: skip the Alpaca news download and the secondary arms (primary result first)
N_JOBS = -1

import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p2, p4, s04, s05, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P4_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p4.P4Config()
WINDOWS = {"A 2020-2025": ("2020-01-01", "2025-12-31"), "B 2017-2019": ("2017-01-01", "2019-12-31")}   # 2026 sealed
QUOTE_CACHE = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
QUOTE_YEARS = ([2017, 2018, 2019], list(range(2020, 2026)))       # the two S05b-method sample draws (P2-04 and S05b)
OUT = ROOT / "notebooks" / "News_Filtered_Reversal" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = work_dir() / "results" / "p4"
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    CFG = replace(CFG, n_placebo_daily=20, n_boot=2000)
    WINDOWS = {"A 2018": ("2018-01-01", "2018-12-31"), "B 2017": ("2017-01-01", "2017-12-31")}
print(f"synthetic={SYNTHETIC} | horizons={CFG.horizons} | alpha={CFG.alpha} | news arm={RUN_NEWS}")

## 1. Daily bars (stocks and sector ETFs), measured costs

In [ ]:
t0 = time.time()
if SYNTHETIC:
    C, V, ETF_OF, EVENTS = p4.synthetic_daily_news()
    STOCKS = sorted(ETF_OF)
    COST = pd.DataFrame(2e-4, index=C.index, columns=STOCKS)
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    ETF_OF = dict(zip(constituents["Symbol"], constituents["GICS Sector"].map(s05.SECTOR_ETF)))
    STOCKS, ETFS = sorted(ETF_OF), sorted(set(ETF_OF.values()))
    YEARS_D = list(range(2016, 2026))
    download_years(STOCKS + ETFS, "1Day", YEARS_D, workers=8)
    C, _, V = p1.load_daily(STOCKS + ETFS, YEARS_D)
    STOCKS = [s for s in STOCKS if s in C.columns and C[s].notna().any()]
    sess = s05b.sample_sessions(QUOTE_YEARS[0], 4, seed=42) + s05b.sample_sessions(QUOTE_YEARS[1], 4, seed=42)
    miss = [d for d in sess if not (QUOTE_CACHE / f"{d}.parquet").exists()]
    assert not miss, f"quote samples missing: {miss} (run S05b and P2-04 first)"
    samples = pd.concat([pd.read_parquet(QUOTE_CACHE / f"{d}.parquet") for d in sess], ignore_index=True)
    COST = p1.daily_cost(C.index, STOCKS, s05b.cost_model(samples))
RET = C[STOCKS].pct_change()
print(f"{len(STOCKS)} stocks, {len(C)} sessions {C.index[0].date()} to {C.index[-1].date()}, ready in {time.time() - t0:.0f}s")

## 2. Earnings events (SEC EDGAR 8-K item 2.02)

In [ ]:
t0 = time.time()
if not SYNTHETIC:
    EVENTS = p4.edgar_earnings(STOCKS, start="2016-01-01", end="2025-12-31")
EF = p4.event_flags(EVENTS, C.index, STOCKS, CFG.win_before, CFG.win_after)
per = EVENTS.assign(y=pd.to_datetime(EVENTS["filing_date"]).dt.year).groupby(["y"]).agg(events=("symbol", "size"), stocks=("symbol", "nunique"))
per["events_per_stock"] = per["events"] / per["stocks"]
display(per); per.to_csv(OUT / "edgar_events_by_year.csv")
print(f"{len(EVENTS):,} earnings 8-Ks for {EVENTS['symbol'].nunique()} stocks; flagged stock-days {EF.to_numpy().mean():.2%}; {time.time() - t0:.0f}s")

## 3. Alpaca news shocks (secondary; resumable download)

In [ ]:
NS = None
if RUN_NEWS:
    t0 = time.time()
    if SYNTHETIC:
        rng = np.random.default_rng(1)
        cnt = pd.DataFrame(rng.poisson(1.0, (len(C), len(STOCKS))), index=C.index, columns=STOCKS)
        cnt = cnt + 6 * EF.astype(int)
    else:
        art = p4.download_news(STOCKS, list(range(2016, 2026)), batch=10)
        cnt = p4.news_counts(art, C.index, STOCKS, CFG.max_tags)
        print(f"{len(art):,} articles; firm-specific per stock-day mean {cnt.to_numpy().mean():.2f}")
    NS = p4.news_shocks(cnt, CFG)
    both = (NS & EF).to_numpy().sum() / max(NS.to_numpy().sum(), 1)
    print(f"news-shock stock-days {NS.to_numpy().mean():.2%}; share of shocks inside earnings windows {both:.1%}; {time.time() - t0:.0f}s")

## 4. Arms by window and horizon

In [ ]:
IND = {}
ELIG0 = (C[STOCKS] >= CFG.min_price)
RES, TAB = {}, []
for k in CFG.horizons:
    lr = np.log(C / C.shift(k))
    sigR = lr[STOCKS]
    etf_lr = pd.DataFrame({s: lr[ETF_OF[s]] for s in STOCKS})
    sigI = sigR - etf_lr
    base = ELIG0 & sigI.notna() & sigR.notna()
    exE = p4.any_in_window(EF, k)
    arms = {"R": (sigR, base), "I": (sigI, base), "IN": (sigI, base & ~exE)}
    if NS is not None:
        exN = p4.any_in_window(NS, k)
        arms["IA"] = (sigI, base & ~exN); arms["INA"] = (sigI, base & ~exE & ~exN)
    IND[k] = {"sigI": sigI, "base": base, "exE": exE}
    for wname, (a, b) in WINDOWS.items():
        win = C.index[(C.index >= a) & (C.index <= b)]
        for arm, (sig, el) in arms.items():
            df = p4.sort_returns(sig, el, RET, COST, k, CFG.n_quantiles, CFG.skip, win)
            RES[(k, wname, arm)] = df
            TAB.append({"k": k, "window": wname, "arm": arm, "gross_bps_day": 1e4 * df.gross.mean(), "cost_bps_day": 1e4 * df.cost.mean(),
                        "net_bps_day": 1e4 * df.net.mean(), "sharpe_gross": np.sqrt(252) * df.gross.mean() / df.gross.std(),
                        "names_per_day": float(el.loc[win].sum(axis=1).mean())})
TAB = pd.DataFrame(TAB).set_index(["k", "window", "arm"])
TAB.to_csv(OUT / "arms.csv", float_format="%.5g")
display(TAB.round(3))
fig, ax = plt.subplots(len(CFG.horizons), len(WINDOWS), figsize=(6 * len(WINDOWS), 3.2 * len(CFG.horizons)), squeeze=False)
ACOL = {"R": "#8a8a84", "I": "#2a78d6", "IN": "#eb6834", "IA": "#1baf7a", "INA": "#7a3fd6"}
for i, k in enumerate(CFG.horizons):
    for j, w in enumerate(WINDOWS):
        for arm in ACOL:
            if (k, w, arm) in RES:
                ax[i, j].plot(RES[(k, w, arm)].gross.cumsum(), color=ACOL[arm], lw=0.9, label=arm)
        ax[i, j].axhline(0, color="#444", lw=0.6); ax[i, j].set_title(f"k = {k}, {w}: cumulative gross")
ax[0, 0].legend(frameon=False); plt.tight_layout(); plt.show()

## 5. Gates

In [ ]:
from joblib import Parallel, delayed
B = lambda x, b: p1.block_bootstrap(np.asarray(x), b, CFG.n_boot, CFG.seed, CFG.alpha)  # noqa: E731
G, STATS, VERD = {}, {}, {}
for k in CFG.horizons:
    G[k] = {}
    for wname, (a, b_) in WINDOWS.items():
        win = C.index[(C.index >= a) & (C.index <= b_)]
        rI, rR, rN = RES[(k, wname, "I")], RES[(k, wname, "R")], RES[(k, wname, "IN")]
        f1b = {b: B(rN.gross - rI.gross, b) for b in CFG.blocks}
        f2b = {b: B(rI.gross - rR.gross, b) for b in CFG.blocks}
        f3b = {b: B(rN.net, b) for b in CFG.blocks}
        pl_path = CACHE / f"{'synthetic_' if SYNTHETIC else ''}placebo_k{k}_{wname.split()[0]}_{CFG.n_placebo_daily}.npy"
        if pl_path.exists():
            dist = np.load(pl_path)
        else:
            t0 = time.time()
            chunks = [(i, min(20, CFG.n_placebo_daily - i)) for i in range(0, CFG.n_placebo_daily, 20)]
            dist = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(p4.placebo_filter)(
                IND[k]["sigI"], IND[k]["base"], IND[k]["exE"], RET, k, CFG, win, n, CFG.seed * 1000 + k * 100 + i) for i, n in chunks))
            np.save(pl_path, dist); print(f"k={k} {wname}: {len(dist)} placebos in {(time.time() - t0) / 60:.1f} min")
        obs = float((rN.gross - rI.gross).mean())
        rk = s04.placebo_rank(obs, dist)
        G[k][wname] = p4.gates_daily(f1b, rk["p_upper"], f2b, f3b, float((rN.gross - 2 * rN.cost).mean()), CFG.alpha)
        STATS[f"{k}|{wname}"] = {"IN_minus_I_bps": {str(b): [1e4 * x for x in v] for b, v in f1b.items()}, "placebo": rk,
                                 "I_minus_R_bps": {str(b): [1e4 * x for x in v] for b, v in f2b.items()},
                                 "IN_net_bps": {str(b): [1e4 * x for x in v] for b, v in f3b.items()}}
        if NS is not None:
            for arm in ("IA", "INA"):
                STATS[f"{k}|{wname}"][f"{arm}_minus_I_bps"] = {str(b): [1e4 * x for x in B(RES[(k, wname, arm)].gross - rI.gross, b)] for b in CFG.blocks}
    VERD[k] = p4.verdict_daily(G[k])
for key, s in STATS.items():
    print(key, "| IN-I", [round(v[0], 3) for v in s["IN_minus_I_bps"].values()][:1], "lo", [round(v[1], 3) for v in s["IN_minus_I_bps"].values()],
          "| placebo pct", round(s["placebo"]["percentile"], 1), "| I-R", round(s["I_minus_R_bps"]["10"][0], 3),
          "lo", [round(v[1], 3) for v in s["I_minus_R_bps"].values()])
display(pd.concat({k: pd.DataFrame(g) for k, g in G.items()})); display(pd.DataFrame(VERD))
summary = {"study": "P4-01", "synthetic": SYNTHETIC, "config": CFG.as_dict(), "windows": WINDOWS, "news_arm": NS is not None,
           "n_stocks": len(STOCKS), "n_events": int(len(EVENTS)), "arms": {"|".join(map(str, i)): v for i, v in TAB.to_dict(orient="index").items()},
           "stats": STATS, "gates": {str(k): g for k, g in G.items()}, "verdicts": {str(k): v for k, v in VERD.items()},
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "p4": p4.P4_VERSION, "p1": p1.P1_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE)}, indent=2))
print(json.dumps({str(k): v for k, v in VERD.items()}, indent=2))